# Series 2.6 — Memory Retrieval

**Why AI Fails? — Engineering Lab**

---

> Building memory is only half of the problem.  
> **Retrieving the right memory** is what makes AI intelligent.

**Scenario:** 100,000 synthetic memory records — four retrieval strategies, precision/recall benchmarks, top-K injection into the prompt.

**Core lesson:** Memory is valuable only when it can be **found**.

## 1. The Problem

| Naive memory search | Intelligent retrieval |
|---------------------|----------------------|
| Keyword match only → misses synonyms | Semantic search finds related concepts |
| Inject entire 100k store into prompt | Top-K relevant memories only |
| No ranking → stale or duplicate facts win | Re-ranking by confidence, recency, priority |
| One strategy for all queries | Hybrid + re-rank for enterprise accuracy |

### Why this matters in production

- Series 2.5 showed **how to build and compress** memory
- At scale, **search quality** determines personalization quality
- Wrong retrieval → wrong prompt context → wrong answers

**Four strategies:**

```
keyword  → inverted index, exact term overlap (fast)
semantic → TF-IDF cosine similarity
hybrid   → keyword + semantic + metadata boost
rerank   → hybrid top-20 → ranking formula → top-K
```

## 2. What is Memory Retrieval?

**Memory retrieval** selects the **most relevant** long-term memories for a user request from a large store — without loading the entire profile into the prompt.

### Definition

```
Memory retrieval = query → intent → search → rank → top-K → prompt
```

### Ranking formula (re-ranking strategy)

```
Memory Score = Semantic Similarity + Confidence + Recency + Business Priority
```

### Dataset characteristics (100k records)

- Categories: User Profile, Policies, Project Knowledge, Security, Database, …
- Includes duplicates, noise, obsolete entries (Flask, MySQL), conflicting preferences
- Tests whether retrieval finds **canonical** facts under realistic messiness

### What memory retrieval is NOT

| Technique | Difference |
|-----------|------------|
| **Memory compression** (2.5) | Compression shrinks the **store**; retrieval selects **what to inject** |
| **RAG chunking** (2.3) | RAG retrieves **documents**; this retrieves **user memories** |
| **Vector DB product** | This lab uses readable keyword + TF-IDF — no black box |

## 3. Repository Layout

```
why-ai-fails/
├── common/
└── series-2.6/
    ├── app.py           ← CLI benchmark entry
    ├── memories.py      ← 100k memory generator
    ├── memory_store.py  ← Inverted index + TF-IDF
    ├── retriever.py     ← Four retrieval strategies
    ├── ranking.py       ← Re-ranking + top-K
    ├── evaluator.py     ← Precision, recall, accuracy
    ├── queries.py       ← Benchmark queries
    ├── benchmark.py
    ├── README.md
    └── Series_2.6_Memory_Retrieval.ipynb   ← This notebook
```

## 4. The Retrieval Pipeline (`retriever.py`)

```
User Request
    ↓
Intent Detection
    ↓
Memory Search (keyword / semantic / hybrid)
    ↓
Memory Ranking (re-rank strategy)
    ↓
Top-K Selection
    ↓
Prompt Builder → Gemini
```

| Strategy | CLI | Candidate pool |
|----------|-----|----------------|
| Keyword | `--strategy keyword` | Inverted index, top 200 |
| Semantic | `--strategy semantic` | TF-IDF sample 5000 |
| Hybrid | `--strategy hybrid` | Combined scoring, top 300 |
| Re-rank | `--strategy rerank` | Hybrid top 20 → full formula |

## 5. Three Layers of Retrieval Engineering

### Layer 1 — Search (cast a wide net)

Keyword search is fast and exact. Semantic search finds related terms (e.g. "postgres" → PostgreSQL). Hybrid combines both.

---

### Layer 2 — Rank (pick the best memories)

Re-ranking boosts confidence, recency, and business priority — so stale Flask preferences don't beat current FastAPI facts.

---

### Layer 3 — Measure (precision, recall, prompt tokens)

| Metric | What it tells you |
|--------|-------------------|
| **Retrieval accuracy** | Expected values in top-K |
| **Precision** | Relevant / retrieved |
| **Recall** | Canonical facts recovered |
| **Prompt tokens** | Only top-K injected — not 100k |

| Mode | Flag | API key? |
|------|------|----------|
| **Dry-run** | `--dry-run` | No — **$0** |
| **Live** | (none) | Yes |

## 6. Execution Flow

```
Parse CLI (--strategy, --query-id, --top-k, --memories)
    │
    └─ Build MemoryStore (default 100,000 records)
            For each benchmark query:
                detect_intent() → retrieve → rank → top-K
                evaluate precision / recall / accuracy
            └─ print_benchmark()
```

## 7. How to Run

From the **repo root**:

```bash
pip install -r requirements.txt
cp .env.example .env   # optional
```

| Command | What it does | API key? |
|---------|--------------|----------|
| `python series-2.6/app.py --dry-run` | All four strategies | No |
| `python series-2.6/app.py --strategy rerank --dry-run` | Best strategy | No |
| `python series-2.6/app.py --memories 10000 --dry-run` | Faster dev test | No |
| `python series-2.6/app.py` | Live Gemini | Yes |

In [ ]:
# Live demo cell — run the dry-run benchmark ($0, no API key needed)
# Execute this cell during your presentation

import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "demo.py").exists() and (ROOT.parent / "demo.py").exists():
    ROOT = ROOT.parent

result = subprocess.run(
    [sys.executable, str(ROOT / "series-2.6/app.py"), "--dry-run"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
print(f"\nExit code: {result.returncode}")


## 8. Key Code Snippets

### Intent detection (`retriever.py`)

```python
def detect_intent(query: str) -> set[str]:
    return set(_tokenize(query))
```

### Strategy names

```python
STRATEGIES = ("keyword", "semantic", "hybrid", "rerank")
RERANK_POOL = 20  # hybrid top-20 before full ranking formula
```

## 9. Where Series 2.6 Fits

| Lab | Topic | Role |
|-----|-------|------|
| 2.5 | Long-Term Memory | Build & compress the store |
| **2.6** | **Memory Retrieval** | **Find** the right memory at scale |
| 2.7 | Model Routing | Route to the right model tier |

---

## Takeaway

> **Store efficiently (2.5). Retrieve precisely (2.6).**

**Next lab:** [Series 2.7 — Model Routing](../series-2.7/) — select the right LLM for each request, not the biggest model every time.